# Flow Matching from Scratch
### The engine behind LongFlow's speed contribution

**Estimated time: ~40 minutes** (reading + running — every cell runs on CPU; a full Run-All takes about 2 minutes, most of it matplotlib warming up).

VibeVoice generates speech one frame at a time, and for **every single frame** it runs a
diffusion head for ~20 denoising steps (times 2 for classifier-free guidance). A 90-minute
podcast is ~40,000 frames. That per-frame loop is the wall LongFlow attacks, and the tool
is **flow matching** — the technique this notebook builds from nothing.

By the end you will have:

1. **The idea** — what a velocity field is and why it beats step-by-step denoising (no code).
2. **A working toy** — a 746-parameter model trained with *our exact loss* on a 2D dataset,
   where you can literally watch noise flow into data and see why 4 steps ≈ 16 steps.
3. **The real code** — `src/flow_head/cfm.py`, line by line, mapped onto the toy.
4. **The real architecture** — the actual `FlowHead` class imported and dissected with tiny tensors.
5. **The payoff** — why our P1 gate hit teacher parity at 4 steps, and how P2 (MeanFlow) aims for 1.
6. **References** — where to go deeper.

Only prerequisite: you can read basic PyTorch. No probability theory required — every
equation that appears also gets said in plain words.


## Part 1 — The problem: turning noise into structured data, on demand

Here is the job, stripped of all TTS detail:

> Given a random noise vector, transform it into a sample from some complicated
> distribution — *conditioned on a hint that says which sample we want.*

In LongFlow's case the "complicated distribution" is 64-dimensional acoustic latents
(one per ~133 ms audio frame), and the hint is the language model's 1536-dim hidden state
for that frame — its "thought vector" encoding what should be spoken, by whom, in what tone.

### Diffusion's answer: many small corrections

Diffusion models learn to *undo* noise a little at a time. To generate, you start from pure
noise and take many small denoising steps, each one asking the network "what does slightly
cleaner data look like from here?" It works beautifully — VibeVoice's own head does exactly
this — but the path from noise to data is **curved**, so you need many steps to follow it
accurately. That's the ~20-step-per-frame cost.

### Flow matching's answer: learn the road, then drive straight

Flow matching asks a different question. Instead of "how do I clean this up a little?", it
learns a **velocity field**:

> at every point in space, at every moment of a 0→1 progress clock, an **arrow**
> saying which way to move and how fast.

Generating is then just: drop a point in the noise, follow the arrows until the clock hits 1.

**The highway metaphor.** Imagine every possible noise sample as a random spot in the
desert, and the data distribution as a city. Diffusion is a nervous sat-nav that
recalculates every few hundred meters along a winding route — safe, but you must check in
constantly. Flow matching instead paves **near-straight highways** from every desert spot
into the city, and the network is just the signpost: "head that way." On a straight road,
checking the signs 4 times gets you essentially where checking 16 times would — and *that*
is the whole speed story.

The one equation of this section, in symbols and in words:

`v = net(x_t, t)` — *"given where I am (`x_t`) and how far along the trip I am (`t`),
tell me the arrow to follow (`v`)."*

Everything else in this notebook is about (a) how to *train* that arrow-printer with a
shockingly simple loss, and (b) how to *use* it with a handful of steps.


## Part 2 — A runnable 2D toy

We'll build the whole pipeline on a 2D dataset first, because in 2D you can *see*
everything: the noise, the data, the learned roads, and the effect of the step budget.
Mentally substitute "2D point" → "64-dim frame latent" and it's the same machine.

**The setup cell below:** imports, a fixed seed, one consistent color per *role* (gray =
noise, blue = real data, red = what the model makes) used in every figure, and — important
for Parts 3–4 — it puts the repo root on `sys.path` so we can import the **real** LongFlow
code (`src/flow_head/...`) later. Works from anywhere inside the repo.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Find the repo root (the folder containing pyproject.toml) so `import src...`
# works no matter which directory Jupyter was started from.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
assert (repo_root / "pyproject.toml").exists(), "run this notebook from inside the LongFlow repo"
sys.path.insert(0, str(repo_root))

torch.manual_seed(0)

# One color per ROLE, reused in every figure of the notebook.
C = {
    "data": "#2a78d6",   # blue - the real dataset (the 'city')
    "noise": "#898781",  # gray - where samples start (the 'desert')
    "model": "#e34948",  # red  - what our trained model produces
}
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.edgecolor": "#c3c2b7", "axes.linewidth": 0.8,
    "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
})
print("repo root:", repo_root)
print("torch", torch.__version__, "- CPU is all we need today")


### The desert and the city

Our stand-in dataset is **two moons**: two interleaved half-circles. It's a good test
because it's *shaped* — a model that just "roughly spreads points around" will visibly fail
to draw the crisp crescents.

The next cell builds it and plots the two ends of our journey: the starting distribution
(pure Gaussian noise, `x0`) and the target distribution (the moons, `x1`). Training a flow
model = learning to transport the left cloud onto the right one.


In [ ]:
def make_moons(n, noise=0.06, generator=None):
    """Two interleaved half-circles - a classic 'shaped' 2D dataset."""
    half = n // 2
    theta = torch.rand(half, generator=generator) * torch.pi
    upper = torch.stack([theta.cos(), theta.sin()], dim=1)
    lower = torch.stack([1 - theta.cos(), -theta.sin() + 0.5], dim=1)
    pts = torch.cat([upper, lower], dim=0)
    pts = pts + noise * torch.randn(pts.shape, generator=generator)
    pts = (pts - torch.tensor([0.5, 0.25])) * 1.6  # center it, spread it out
    return pts[torch.randperm(n, generator=generator)]  # shuffle the two moons together

g = torch.Generator().manual_seed(0)
data = make_moons(4096, generator=g)         # x1: 4096 'clean' samples
noise0 = torch.randn(1500, 2, generator=g)   # x0: where every sample will start

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
axes[0].scatter(noise0[:, 0], noise0[:, 1], s=4, color=C["noise"], alpha=0.5)
axes[0].set_title("Where we start: noise  x0 ~ N(0, I)")
axes[1].scatter(data[:1500, 0], data[:1500, 1], s=4, color=C["data"], alpha=0.5)
axes[1].set_title("Where we want to end: data  x1")
for ax in axes:
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()


### The model: a signpost printer

The network's only job: take a position and the progress clock, return an arrow.
Input `(x, y, t)` — three numbers. Output `(vx, vy)` — two numbers. We make it
deliberately tiny (**746 parameters**) to drive home that the *technique*, not model
capacity, does the heavy lifting here.

Note the interface — `velocity = net(x_t, t)`. It's the same interface as the real
`FlowHead` in Part 4, which just adds one more input (the conditioning hint).


In [ ]:
class TinyVelocityNet(torch.nn.Module):
    """(x, y, t) -> (vx, vy): 'standing here, at this time, drive that way'."""

    def __init__(self, hidden=24):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(3, hidden), torch.nn.SiLU(),
            torch.nn.Linear(hidden, hidden), torch.nn.SiLU(),
            torch.nn.Linear(hidden, 2),
        )

    def forward(self, x_t, t):
        return self.net(torch.cat([x_t, t[:, None]], dim=1))

net = TinyVelocityNet()
print("parameters:", sum(p.numel() for p in net.parameters()))


### The training loss — the entire trick in 4 lines

This is a faithful copy of the loss in `src/flow_head/cfm.py` (the "OT-CFM" /
linear-interpolant objective used by F5-TTS, ZipVoice, and friends):

```python
t        = torch.rand(b)                    # 1. pick a random moment on the 0->1 clock
x0       = torch.randn_like(x1)             # 2. pick a random noise point
x_t      = (1 - t) * x0 + t * x1            # 3. stand t of the way along the STRAIGHT
                                            #    line from that noise to a data point
v_target = x1 - x0                          # 4. the arrow that line follows
loss     = MSE(net(x_t, t), v_target)       #    ...and ask the net to predict it
```

In words: *pair a random noise point with a random data point, walk partway along the
straight line between them, and train the network to point along that line.* Notice the
target `x1 - x0` doesn't depend on `t` — a straight line has the same direction everywhere
along it. That's the "highways are straight" property, built directly into the training
signal.

**The one honest subtlety** (worth 30 seconds): the pairing is *random* — nothing says this
noise point "belongs" to that data point, and many different lines pass near any given
`(x_t, t)`. So the target is ambiguous, and an MSE-trained net learns the **average** arrow
at each point. The theoretical result that makes flow matching work (Lipman et al., 2022 —
see Part 6) is that following this *averaged* field still transports the noise distribution
**exactly** onto the data distribution. You don't have to take that on faith — the plots
below are the demonstration.

A visible consequence: **the loss will plateau well above zero** (~1.6 here). That's the
irreducible pairing ambiguity being averaged out, not underfitting. Don't chase it to zero.


In [ ]:
def toy_cfm_loss(net, x1, generator=None):
    """Same 4 lines as cfm_loss() in src/flow_head/cfm.py, minus the condition."""
    t = torch.rand(x1.shape[0], generator=generator)
    x0 = torch.randn(x1.shape, generator=generator)
    x_t = (1 - t[:, None]) * x0 + t[:, None] * x1
    v_target = x1 - x0
    return torch.nn.functional.mse_loss(net(x_t, t), v_target)

opt = torch.optim.Adam(net.parameters(), lr=1e-2)
losses = []
for step in range(2500):  # ~1 second on CPU
    idx = torch.randint(0, data.shape[0], (512,), generator=g)
    loss = toy_cfm_loss(net, data[idx], generator=g)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

smooth = torch.tensor(losses).unfold(0, 25, 1).mean(1)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses, color=C["model"], alpha=0.25, linewidth=1)
ax.plot(range(24, len(losses)), smooth, color=C["model"], linewidth=2)
ax.set_title("Training loss (raw + 25-step average) - plateaus by design, not failure")
ax.set_xlabel("step")
ax.set_ylabel("MSE")
plt.tight_layout()
plt.show()
print(f"final smoothed loss: {smooth[-1]:.3f}  (the plateau IS the pairing ambiguity)")


### Sampling = following the arrows (the Euler loop)

To generate: drop a point in the noise, then repeat a few times — *ask the net for the
arrow, drive along it for a fraction of the clock*:

```python
x = x + dt * net(x, t)      # "drive in the pointed direction for dt of the trip"
```

That's Euler integration, the simplest ODE solver there is. **NFE** ("number of function
evaluations") = how many times you stop to check the signposts. NFE 16 means 16 small
steps; NFE 1 means one blind jump using only the arrow you read at the start.

The next cell draws the actual **trajectories** — 120 points travelling from noise (gray
dots) to their landing spots (red), over the real data (faint blue). Watch two things:

- with enough checks (NFE 16), the roads are short and only **gently curved** — most of
  each trip is a straight run at the target. That's what makes coarse stepping an option.
- at NFE 1 every point takes one blind straight jump. It lands *near* the data, but where
  the road curved, it lands slightly wrong — and where the field is still ambiguous (deep
  in the noise, between the moons), it can't commit to either moon at all.


In [ ]:
@torch.no_grad()
def toy_euler_sample(net, n, nfe, generator=None, keep_path=False):
    """Mirror of euler_sample() in src/flow_head/cfm.py (uniform grid, no condition)."""
    grid = torch.linspace(0, 1, nfe + 1)
    x = torch.randn(n, 2, generator=generator)
    path = [x.clone()]
    for i in range(nfe):
        t = grid[i].expand(n)
        x = x + (grid[i + 1] - grid[i]) * net(x, t)
        path.append(x.clone())
    return (x, torch.stack(path)) if keep_path else x

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True)
for ax, nfe in zip(axes, (1, 4, 16)):
    _, path = toy_euler_sample(net, 120, nfe, torch.Generator().manual_seed(7), keep_path=True)
    ax.scatter(data[:1200, 0], data[:1200, 1], s=3, color=C["data"], alpha=0.15)
    ax.plot(path[:, :, 0], path[:, :, 1], color=C["model"], alpha=0.2, linewidth=0.7)
    ax.scatter(path[0, :, 0], path[0, :, 1], s=5, color=C["noise"], alpha=0.7, label="start (noise)")
    ax.scatter(path[-1, :, 0], path[-1, :, 1], s=7, color=C["model"], alpha=0.9, label="end (sample)")
    ax.set_title(f"NFE = {nfe}" + ("  (one straight jump)" if nfe == 1 else ""))
    ax.set_aspect("equal")
axes[0].legend(loc="lower left", fontsize=8)
fig.suptitle("Trajectories from noise to data - mostly straight roads, so few checks suffice", y=1.02)
plt.tight_layout()
plt.show()


### The money plot: same noise, four step budgets

Now the experiment that motivates the whole project. We generate 2,000 samples from the
**same starting noise** at NFE 1, 2, 4, and 16, and print in each panel's title the mean
distance from a generated point to its nearest real data point (lower = better).

What to look for:

- **NFE 1** — everything lands in a clump near the center. One arrow, read while still
  deep in the noise, is the *average* over both moons — it points at "the middle of the
  city", not at either moon. One check simply can't commit to a destination.
- **NFE 2** — a dramatic jump: the second check, taken halfway through the trip, lets each
  point commit. The moons appear, still smeared.
- **NFE 4 vs NFE 16** — squint. That small difference is the whole point: once the road is
  mostly straight, the extra 12 network calls buy almost nothing. This is the toy version
  of the gate result in `experiments/p1_flow_head/NOTES.md` (Part 5).


In [ ]:
def mean_nn_dist(samples, data):
    """Mean distance from each sample to its nearest real data point."""
    return torch.cdist(samples, data).min(dim=1).values.mean().item()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), sharex=True, sharey=True)
for ax, nfe in zip(axes, (1, 2, 4, 16)):
    samples = toy_euler_sample(net, 2000, nfe, torch.Generator().manual_seed(42))
    ax.scatter(data[:1500, 0], data[:1500, 1], s=3, color=C["data"], alpha=0.12)
    ax.scatter(samples[:, 0], samples[:, 1], s=3, color=C["model"], alpha=0.45)
    ax.set_title(f"NFE = {nfe}\nmean dist to data = {mean_nn_dist(samples, data):.3f}")
    ax.set_aspect("equal")
fig.suptitle("Same starting noise, different step budgets (red = generated, blue = real)", y=1.06)
plt.tight_layout()
plt.show()


## Part 3 — The real code, line by line

You have now written flow matching. Here is the production version, `cfm_loss` from
`src/flow_head/cfm.py`, in full:

```python
def cfm_loss(head, x1: torch.Tensor, condition: torch.Tensor, generator=None) -> torch.Tensor:
    """x1: clean latents [B, d_latent]; condition: [B, d_model]."""
    if not torch.isfinite(x1).all() or not torch.isfinite(condition).all():
        raise ValueError("non-finite training batch - refusing to step")
    b = x1.shape[0]
    t = torch.rand(b, device=x1.device, generator=generator)
    x0 = torch.randn(x1.shape, device=x1.device, dtype=x1.dtype, generator=generator)
    x_t = (1 - t[:, None]) * x0 + t[:, None] * x1
    v_target = x1 - x0
    v_pred = head(x_t, t, condition)
    return torch.nn.functional.mse_loss(v_pred, v_target)
```

Mapping onto the toy, line by line:

| line | toy equivalent | what's different, and why |
|---|---|---|
| `isfinite` guard | (none) | production hygiene: one NaN latent in a cached batch must halt training loudly, not poison the head silently |
| `t = torch.rand(b, ...)` | same | one random clock time **per batch element**, uniform on [0, 1] |
| `x0 = torch.randn(...)` | same | fresh noise, matched to `x1`'s device/dtype; `generator` makes runs reproducible |
| `x_t = (1-t)*x0 + t*x1` | same | `t[:, None]` broadcasts the `[B]` clock across the 64 latent dims |
| `v_target = x1 - x0` | same | the straight-line arrow, constant in `t` |
| `head(x_t, t, condition)` | `net(x_t, t)` | **the one real difference**: the head also sees the 1536-dim hint (Part 4) |

And the sampler, `euler_sample` — again, you already wrote this:

```python
@torch.no_grad()
def euler_sample(head, condition, d_latent, nfe=4, sway=0.0, generator=None):
    """condition [B, d_model] -> sampled latents [B, d_latent]."""
    b = condition.shape[0]
    grid = sway_grid(nfe, sway, device=condition.device)
    x = torch.randn(b, d_latent, device=condition.device, generator=generator)
    for i in range(nfe):
        t = grid[i].expand(b)
        v = head(x, t, condition)
        x = x + (grid[i + 1] - grid[i]) * v
    if not torch.isfinite(x).all():
        raise RuntimeError("non-finite sample - head diverged")
    return x
```

- `grid = sway_grid(nfe, sway, ...)` — the time grid, possibly **warped** (next section).
  Because the grid may be non-uniform, the step size is `grid[i+1] - grid[i]` rather than
  a constant `1/nfe`.
- `x = torch.randn(b, d_latent, ...)` — every frame's journey starts from fresh noise.
- the loop — exactly your toy loop: read the arrow, drive, repeat `nfe` times.
- the finite check — if the head ever diverges, fail loudly at the sample, not three
  stages later inside the vocoder.


### Sway sampling: spend your few steps where they matter

A uniform grid checks the signposts at evenly spaced times. But the trip isn't equally
tricky everywhere: **early in the trip (small `t`, still deep in the noise) is where a
trajectory commits to its destination** — which moon, which mode. Late in the trip you're
on final approach and the arrows barely change.

F5-TTS's *sway sampling* warps the grid accordingly:

```python
u + coef * (cos(pi/2 * u) - 1 + u)     # coef < 0 pushes grid points toward t = 0
```

In words: *take the uniform grid and slide its points toward the start of the trip, so a
small step budget spends most of its checks in the decision-heavy early phase.* Same
trained model, same NFE — just smarter timing. A pure inference-time knob.

The next cell does two things with the **real imported functions**:

1. plots uniform vs. swayed grid points for NFE 8, so you can see the warp;
2. runs the **actual `euler_sample` from `src/flow_head/cfm.py`** on our toy net (wrapped
   to ignore the condition) — first checking it reproduces the hand-rolled sampler exactly
   at `sway=0`, then sampling at `sway=-1.0`.

Honest footnote: on this toy, sway actually *hurts* a little (you'll see the numbers) —
the warp is tuned for the speech setting, where F5-TTS found it helps at low NFE. That's
exactly why it's an inference-time *option* in our sampler, defaulting to off: cheap to
try, easy to A/B per task, no retraining.


In [ ]:
from src.flow_head.cfm import euler_sample, sway_grid  # <- the real project code

# --- 1. what the warp does to the grid
fig, ax = plt.subplots(figsize=(8, 1.8))
for y, (label, coef) in enumerate([("sway  coef=-1.0", -1.0), ("uniform", 0.0)]):
    pts = sway_grid(8, coef)
    ax.scatter(pts, [y] * len(pts), s=40, color=C["model"] if coef else C["noise"], zorder=3)
    ax.text(-0.05, y, label, ha="right", va="center", fontsize=9)
ax.annotate("more checks early, where trajectories commit",
            xy=(0.17, 0.12), xytext=(0.4, 0.45), fontsize=8,
            arrowprops=dict(arrowstyle="->", color="#52514e"))
ax.set_xlim(-0.45, 1.05)
ax.set_ylim(-0.5, 1.5)
ax.set_yticks([])
ax.set_xlabel("t (progress clock)")
ax.set_title("The same 9 grid points, uniform vs. swayed (NFE 8)")
plt.tight_layout()
plt.show()

# --- 2. the REAL sampler, driving the toy net
class IgnoreCondition(torch.nn.Module):
    """Adapter: euler_sample passes (x, t, condition); the toy net takes no condition."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, x_t, t, condition):
        return self.net(x_t, t)

wrapped = IgnoreCondition(net)
cond = torch.zeros(2000, 1)  # dummy hint; its width is arbitrary here

ours = toy_euler_sample(net, 2000, nfe=4, generator=torch.Generator().manual_seed(42))
real = euler_sample(wrapped, cond, d_latent=2, nfe=4, sway=0.0,
                    generator=torch.Generator().manual_seed(42))
assert torch.allclose(ours, real), "project sampler must equal the hand-rolled loop"
print("sway=0: real euler_sample == our toy loop, bit for bit. It IS the same algorithm.")

swayed = euler_sample(wrapped, cond, d_latent=2, nfe=4, sway=-1.0,
                      generator=torch.Generator().manual_seed(42))
print(f"NFE 4 mean dist to data:  uniform {mean_nn_dist(real, data):.3f}   "
      f"sway -1.0 {mean_nn_dist(swayed, data):.3f}")


## Part 4 — Conditioning, and the real `FlowHead` in plain terms

The toy learned **one** fixed distribution. The real head has a harder brief: for *every
frame*, produce a latent from a **different** distribution — the one described by the hint.
The hint is VibeVoice's last hidden state for that frame, a 1536-dim "thought vector"
carrying everything the language model knows about what the frame should sound like:
phonetics, speaker, prosody, context.

Mechanically, conditioning changes almost nothing: same loss, same sampler — the network
simply takes the hint as a third input, `v = head(x_t, t, condition)`, and learns
*"given this thought vector, where do plausible latents live, and which way is it from
here?"* Hold that phrase; it's the key to Part 5.

### The architecture, ingredient by ingredient

`FlowHead` (`src/flow_head/model.py`) is a stack of 4 identical blocks, width 640, ~16.6M
parameters, **no attention** — the backbone already did all the cross-frame reasoning, so
the head works one frame at a time. It mirrors VibeVoice's own diffusion-head design,
shrunk 8x and retargeted from diffusion to velocity prediction. The ingredients:

- **RMSNorm** — a volume normalizer. It rescales each activation vector to a standard
  length before it's processed (like normalizing audio levels before mixing), which keeps
  deep stacks stable. "RMS" because it divides by the root-mean-square of the vector's
  entries.

- **SwiGLU** — a *gated* feedforward layer. Two parallel linear projections look at the
  input: one proposes content (`up`), the other decides how much of each channel gets
  through (`gate`, squashed by the smooth SiLU curve). Multiply the two, project back
  down. The gate lets the layer say "for this input, mute those features" — cheap, and it
  reliably beats a plain MLP.

- **AdaLN-zero** — *how the condition steers the network.* The hint doesn't enter as extra
  input features. Instead, each block computes three vectors from it — `shift`, `scale`,
  `gate` — and uses them to modulate that block's activations:

  ```python
  h = norm(x) * (1 + scale) + shift    # the condition re-tunes what the block sees
  x = x + gate * ffn(h)                # ...and how loudly the block speaks
  ```

  The "-zero" part: those modulation weights **start at exactly zero**, and so does the
  final output projection. So at initialization every block is a perfect pass-through and
  the whole head outputs exactly **0** — a known-safe no-op. Training then *gradually
  turns the steering up* from a stable start instead of from random garbage. (We verify
  the exact-zero claim on the real class below.)

- **Timestep embedding** — how the net reads the clock. A raw scalar `t` is hard for a
  network to use precisely, so it's expanded into a bank of sines and cosines at many
  frequencies — a clock with many hands, from very fast to very slow, giving every moment
  a rich, distinctive fingerprint. (Same trick as transformer position embeddings.) This
  embedding is summed with the projected hint, and that sum is what drives every block's
  AdaLN steering.

The next cell imports the **real class** and walks the shapes with doll-house dimensions,
then instantiates the true P1 configuration to check the parameter count against
`experiments/p1_flow_head/NOTES.md`.


In [ ]:
from src.flow_head.model import FlowHead, FlowHeadConfig, timestep_embedding

# Doll-house dims so every tensor is small enough to look at.
tiny = FlowHead(FlowHeadConfig(d_model=32, d_latent=8, width=16, layers=2))
print(f"doll-house FlowHead: {tiny.param_count():,} params")

B = 5
x_t = torch.randn(B, 8)     # noisy latents   [B, d_latent]
t = torch.rand(B)           # clock times     [B]
cond = torch.randn(B, 32)   # thought vectors [B, d_model]

emb = timestep_embedding(t, 16)
print("timestep_embedding(t, width):", tuple(emb.shape), "- sines+cosines at 8 frequencies")

v = tiny(x_t, t, cond)
print("velocity out:", tuple(v.shape), "- same shape as x_t, as a velocity must be")
print("max |v| at init:", v.abs().max().item(), "<- AdaLN-zero + zero out_proj = exact no-op start")

# The real P1 configuration (width 640 and 4 layers are the defaults).
real_head = FlowHead(FlowHeadConfig(d_model=1536, d_latent=64))
print(f"real P1 config: {real_head.param_count() / 1e6:.2f}M params "
      f"(NOTES.md gate run: 16.64M; VibeVoice's own diffusion head: 123M)")


## Part 5 — Why 4 steps is enough *here*, and the P2 preview

Two things compound in LongFlow's favor:

1. **Flow-matching roads are trained straight-ish** — you watched that in the trajectory
   plot. Coarse Euler stepping on near-straight paths loses very little.
2. **Our conditioning is unusually strong.** The toy had to sculpt two sprawling moons out
   of noise with *no hint at all*. The real head gets a per-frame 1536-dim thought vector
   that very nearly *determines* the answer — the conditional distribution "latents
   consistent with this exact hint" is a small, tight blob, not a moons-shaped sprawl.
   Short, straight trips: the easiest possible job for a few-step sampler.

The pre-registered P1 gate (`experiments/p1_flow_head/NOTES.md`) confirmed it. A
16.64M-param head, trained with exactly the `cfm_loss` above on cached VibeVoice hidden
states from 800 utterances, decoded through the frozen VAE:

| config | WER vs teacher | speaker similarity to teacher |
|---|---|---|
| **flow, 4 NFE** | **0.030** | **0.984** |
| flow, 16 NFE | 0.030 | 0.978 |

Four steps didn't just *approach* sixteen — they were **indistinguishable** (durations
identical, F0 within a few Hz; Josh's ears agreed). 4 NFE already saturates this head,
against a teacher spending ~20 diffusion-solver steps x2 CFG passes per frame. That
saturation is the green light for the next push.

### P2 preview: MeanFlow — or, why stop at 4?

Everything above learns the **instantaneous** velocity: the arrow *right here, right now*.
Even on a curved road you could jump start-to-finish in one step *if* you knew the
**average** velocity over the whole trip. That is MeanFlow (Geng et al., 2025): the network
learns `u(x, r, t)` — "my average direction and speed over the interval from time `r` to
time `t`" — so a single evaluation at `(r=0, t=1)` jumps from noise straight to the
target. One NFE, trained from scratch, no separate distillation stage required.

The catch is training it. The average velocity has to stay consistent with the
instantaneous one, and the *MeanFlow identity* that enforces this involves a
time-derivative of the network itself. That derivative comes almost for free from
forward-mode autodiff — `torch.func.jvp`, the "JVP trick" (arXiv:2505.13447) — but the
recipe is booby-trapped: the wrong JVP tangent fails silently and catastrophically, and
the time-embedding must live *inside* the differentiated closure. We won't derive any of
it here; the vetted recipe — and the fallback plan of distilling from the P1 head if
MeanFlow-from-scratch misbehaves — is written down in `docs/resources.md` section 2. Which
is exactly why the P1 4-NFE head is kept in good shape: it's both the baseline and the
safety net.


## Part 6 — Go deeper

In a sensible reading order:

1. **Fjelde, Mathieu & Dutordoir — "An Introduction to Flow Matching"** (Cambridge MLG
   blog, 2024). The gentlest full treatment, with interactive figures — the best next step
   after this notebook.
   [mlg.eng.cam.ac.uk/blog/2024/01/20/flow-matching.html](https://mlg.eng.cam.ac.uk/blog/2024/01/20/flow-matching.html)
2. **Holderrieth & Erives — "An Introduction to Flow Matching and Diffusion Models"**
   (MIT 6.S184 lecture notes + videos): a full course building both frameworks side by
   side. [diffusion.csail.mit.edu](https://diffusion.csail.mit.edu) ·
   [arXiv:2506.02070](https://arxiv.org/abs/2506.02070)
3. **Lipman et al. — "Flow Matching for Generative Modeling"** — the original paper; the
   averaged-field result from Part 2 lives here.
   [arXiv:2210.02747](https://arxiv.org/abs/2210.02747)
4. **Tong et al. — "Improving and generalizing flow-based generative models with minibatch
   optimal transport"** — where the "OT-CFM" framing comes from, plus the reference
   library: [arXiv:2302.00482](https://arxiv.org/abs/2302.00482) ·
   [github.com/atong01/conditional-flow-matching](https://github.com/atong01/conditional-flow-matching)
   (we hand-roll instead — `docs/resources.md` section 2 explains why).
5. **F5-TTS** — flow matching for TTS at scale, and the source of sway sampling (Part 3).
   [arXiv:2410.06885](https://arxiv.org/abs/2410.06885)
6. **ZipVoice** — the same lineage pushed hard on efficiency; the closest published
   relative of our head's design goals. [arXiv:2506.13053](https://arxiv.org/abs/2506.13053)
7. **MeanFlow — "Mean Flows for One-step Generative Modeling"** — the P2 plan.
   [arXiv:2505.13447](https://arxiv.org/abs/2505.13447)

And in this repo: `src/flow_head/cfm.py` (you've now read every line of it),
`src/flow_head/model.py`, `docs/resources.md` section 2 (the researched recipe, including
the MeanFlow traps), and `experiments/p1_flow_head/NOTES.md` (the gate protocol and
result).
